# TNBC Parameter Sweep

Parallel parameter exploration for model sensitivity and robustness. Results are computational hypotheses and require experimental validation.

In [ ]:
import sys, os, itertools
sys.path.insert(0, '/content/TNBC-Metabolic-Strain-MOD')
from src.tnbc_model import simulate, DEFAULT_CONTROL
import numpy as np
from concurrent.futures import ProcessPoolExecutor

def run_one(item):
    name, value = item
    p = DEFAULT_CONTROL.copy(); p[name] = float(value)
    t,y = simulate(p)
    return name, value, float(y[-1,0]), float(y[:,1].max())

grid = {
    'k_glyc': np.linspace(0.3, 0.9, 7),
    's': np.linspace(0.05, 0.30, 6),
    'g': np.linspace(0.10, 0.40, 7),
}
tasks = [(k,v) for k,vals in grid.items() for v in vals]

with ProcessPoolExecutor() as ex:
    results = list(ex.map(run_one, tasks))

import pandas as pd
df = pd.DataFrame(results, columns=['parameter','value','final_ATP','max_ROS'])
display(df.head())
